# Pseudo-labeling and controlled ablation

This notebook applies the tuned gold-only Logistic Regression model to BanglishRev, enforces deterministic confidence and class-balance caps, retrains identical model configurations, and compares them on gold validation data.


In [1]:
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy>=1.26.0",
    "pandas": "pandas>=2.1.0",
    "pyspark": "pyspark>=3.5.0,<4.0",
    "sklearn": "scikit-learn>=1.3.0",
}
missing_packages = [
    package
    for module, package in required_packages.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing_packages],
        check=True,
    )
    importlib.invalidate_caches()

import json
import math
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from pyspark.ml.classification import (
    GBTClassificationModel, GBTClassifier,
    LinearSVC, LinearSVCModel,
    LogisticRegression, LogisticRegressionModel,
    NaiveBayes, NaiveBayesModel,
    RandomForestClassificationModel, RandomForestClassifier
)
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.functions import array_to_vector, vector_to_array
from pyspark.sql import SparkSession, Window, functions as F
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score
)

SEED = 42
CONFIDENCE_THRESHOLD = 0.90
SENSITIVITY_THRESHOLD = 0.95
PSEUDO_SOURCE_WEIGHT = 0.50
MAX_PSEUDO_GOLD_RATIO = 5.0
MAX_PSEUDO_CLASS_SHARE = 0.80

BASE_DIR = Path("/kaggle/working/bangla_fake_review")
PROCESSED_DIR = BASE_DIR / "processed"
MODELS_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"
AUDIT_DIR = BASE_DIR / "audit"
for directory in (PROCESSED_DIR, MODELS_DIR, RESULTS_DIR, AUDIT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("BanglaFakeReview_PseudoLabeling")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "32")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

def resolve_artifact_directory(
    preferred, required_names, source_label, input_root=Path("/kaggle/input")
):
    if all((preferred / name).exists() for name in required_names):
        return preferred
    candidates = []
    if input_root.exists():
        for first_artifact in input_root.rglob(required_names[0]):
            candidate = first_artifact.parent
            if all((candidate / name).exists() for name in required_names):
                candidates.append(candidate)
    candidates = sorted(set(candidates))
    if not candidates:
        raise FileNotFoundError(
            f"Attach the saved output of {source_label} as a Kaggle input."
        )
    if len(candidates) > 1:
        raise RuntimeError(
            f"Multiple compatible outputs found for {source_label}: {candidates}"
        )
    return candidates[0]

stage_three_files = [
    "train_model_ready.parquet",
    "validation_model_ready.parquet",
    "banglish_model_ready.parquet",
]
gold_model_names = [
    "logistic_regression_gold",
    "naive_bayes_gold",
    "linear_svc_gold",
    "random_forest_gold",
    "gbt_gold",
]
INPUT_PROCESSED_DIR = resolve_artifact_directory(
    PROCESSED_DIR, stage_three_files, "notebook 03"
)
GOLD_MODELS_DIR = resolve_artifact_directory(
    MODELS_DIR, gold_model_names, "notebook 04"
)
print(f"Reading notebook 03 artifacts from {INPUT_PROCESSED_DIR}")
print(f"Reading notebook 04 models from {GOLD_MODELS_DIR}")

train = spark.read.parquet(
    str(INPUT_PROCESSED_DIR / "train_model_ready.parquet")
)
validation = spark.read.parquet(
    str(INPUT_PROCESSED_DIR / "validation_model_ready.parquet")
)
banglish = spark.read.parquet(
    str(INPUT_PROCESSED_DIR / "banglish_model_ready.parquet")
)

nb_feature_assembler = VectorAssembler(
    inputCols=["term_frequency", "handcrafted_nonnegative"],
    outputCol="features_nb",
    handleInvalid="error",
)

def sanitize_naive_bayes_features(frame):
    values = vector_to_array("handcrafted_nonnegative")
    cleaned_values = F.transform(
        values,
        lambda value: F.when(
            value.isNull()
            | F.isnan(value)
            | (value == F.lit(float("inf")))
            | (value == F.lit(float("-inf"))),
            F.lit(0.0),
        ).otherwise(F.greatest(value, F.lit(0.0))),
    )
    prepared = (
        frame.drop("features_nb")
        .withColumn(
            "handcrafted_nonnegative",
            array_to_vector(cleaned_values),
        )
    )
    return nb_feature_assembler.transform(prepared)

train = sanitize_naive_bayes_features(train)
validation = sanitize_naive_bayes_features(validation)
banglish = sanitize_naive_bayes_features(banglish)

gold_models = {
    "logistic_regression": LogisticRegressionModel.load(
        str(GOLD_MODELS_DIR / "logistic_regression_gold")
    ),
    "naive_bayes": NaiveBayesModel.load(
        str(GOLD_MODELS_DIR / "naive_bayes_gold")
    ),
    "linear_svc": LinearSVCModel.load(
        str(GOLD_MODELS_DIR / "linear_svc_gold")
    ),
    "random_forest": RandomForestClassificationModel.load(
        str(GOLD_MODELS_DIR / "random_forest_gold")
    ),
    "gbt": GBTClassificationModel.load(str(GOLD_MODELS_DIR / "gbt_gold")),
}


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/20 15:13:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Reading notebook 03 artifacts from /kaggle/input/notebooks/shuhridabrar/feature-engineering/bangla_fake_review/processed
Reading notebook 04 models from /kaggle/input/notebooks/shuhridabrar/04-gold-only-training/bangla_fake_review/models


In [2]:
def evaluate_predictions(predictions, model_name):
    columns = ["label", "prediction"]
    if "probability" in predictions.columns:
        score_column = "probability"
        columns.append(score_column)
    else:
        score_column = "rawPrediction"
        columns.append(score_column)

    rows = predictions.select(*columns).collect()
    labels = np.asarray([int(row["label"]) for row in rows])
    predicted = np.asarray([int(row["prediction"]) for row in rows])
    fake_target = (labels == 0).astype(int)
    fake_score = np.asarray([float(row[score_column][0]) for row in rows])

    return {
        "model": model_name,
        "macro_f1": float(f1_score(labels, predicted, average="macro", zero_division=0)),
        "macro_precision": float(
            precision_score(labels, predicted, average="macro", zero_division=0)
        ),
        "macro_recall": float(
            recall_score(labels, predicted, average="macro", zero_division=0)
        ),
        "accuracy": float(accuracy_score(labels, predicted)),
        "fake_pr_auc": float(average_precision_score(fake_target, fake_score)),
        "fake_roc_auc": float(roc_auc_score(fake_target, fake_score)),
        "fake_f1": float(
            f1_score(labels, predicted, labels=[0], average="macro", zero_division=0)
        ),
        "genuine_f1": float(
            f1_score(labels, predicted, labels=[1], average="macro", zero_division=0)
        ),
        "confusion_matrix": confusion_matrix(labels, predicted, labels=[0, 1]).tolist(),
    }


def model_parameter(model, name):
    return model.getOrDefault(model.getParam(name))


def estimator_from_gold(name, model):
    common = {"labelCol": "label", "weightCol": "sample_weight"}
    if name == "logistic_regression":
        return LogisticRegression(
            featuresCol="features_linear",
            maxIter=int(model_parameter(model, "maxIter")),
            regParam=float(model_parameter(model, "regParam")),
            elasticNetParam=float(model_parameter(model, "elasticNetParam")),
            family="binomial",
            **common,
        )
    if name == "naive_bayes":
        return NaiveBayes(
            featuresCol="features_nb",
            smoothing=float(model_parameter(model, "smoothing")),
            modelType="multinomial",
            **common,
        )
    if name == "linear_svc":
        return LinearSVC(
            featuresCol="features_linear",
            maxIter=int(model_parameter(model, "maxIter")),
            regParam=float(model_parameter(model, "regParam")),
            **common,
        )
    if name == "random_forest":
        return RandomForestClassifier(
            featuresCol="features_tree",
            numTrees=int(model_parameter(model, "numTrees")),
            maxDepth=int(model_parameter(model, "maxDepth")),
            minInstancesPerNode=int(model_parameter(model, "minInstancesPerNode")),
            featureSubsetStrategy=str(model_parameter(model, "featureSubsetStrategy")),
            seed=SEED,
            **common,
        )
    if name == "gbt":
        return GBTClassifier(
            featuresCol="features_tree",
            maxIter=int(model_parameter(model, "maxIter")),
            maxDepth=int(model_parameter(model, "maxDepth")),
            minInstancesPerNode=int(model_parameter(model, "minInstancesPerNode")),
            stepSize=float(model_parameter(model, "stepSize")),
            seed=SEED,
            **common,
        )
    raise ValueError(f"Unsupported model: {name}")


In [3]:
pseudo_predictions = (
    gold_models["logistic_regression"].transform(banglish)
    .withColumn("_probability_values", vector_to_array("probability"))
    .withColumn("probability_fake", F.col("_probability_values")[0])
    .withColumn("probability_genuine", F.col("_probability_values")[1])
    .withColumn(
        "confidence_score",
        F.greatest("probability_fake", "probability_genuine"),
    )
    .withColumn("pseudo_label", F.col("prediction").cast("int"))
    .drop("_probability_values")
)

def deterministic_cap(frame, threshold):
    accepted = frame.filter(F.col("confidence_score") > threshold)
    counts = {
        int(row["pseudo_label"]): int(row["count"])
        for row in accepted.groupBy("pseudo_label").count().collect()
    }
    counts.setdefault(0, 0)
    counts.setdefault(1, 0)
    gold_count = train.count()
    maximum_total = int(MAX_PSEUDO_GOLD_RATIO * gold_count)

    if counts[0] == 0 or counts[1] == 0:
        limits = {0: 0, 1: 0}
    else:
        minority_label = min(counts, key=counts.get)
        majority_label = 1 - minority_label
        limits = dict(counts)
        maximum_majority = math.floor(
            counts[minority_label]
            * MAX_PSEUDO_CLASS_SHARE
            / (1.0 - MAX_PSEUDO_CLASS_SHARE)
        )
        limits[majority_label] = min(limits[majority_label], maximum_majority)

        capped_total = sum(limits.values())
        if capped_total > maximum_total:
            scale = maximum_total / capped_total
            limits = {
                label: max(1, math.floor(limit * scale))
                for label, limit in limits.items()
            }

    order_window = Window.partitionBy("pseudo_label").orderBy(
        F.sha2(
            F.concat_ws("|", F.col("review_id"), F.lit(str(SEED))),
            256,
        )
    )
    capped = (
        accepted.withColumn("_rank", F.row_number().over(order_window))
        .filter(
            ((F.col("pseudo_label") == 0) & (F.col("_rank") <= limits[0]))
            | ((F.col("pseudo_label") == 1) & (F.col("_rank") <= limits[1]))
        )
        .drop("_rank")
    )
    capped_counts = {
        int(row["pseudo_label"]): int(row["count"])
        for row in capped.groupBy("pseudo_label").count().collect()
    }
    return capped, {
        "threshold": threshold,
        "accepted_before_cap": sum(counts.values()),
        "class_counts_before_cap": counts,
        "class_limits": limits,
        "accepted_after_cap": capped.count(),
        "class_counts_after_cap": capped_counts,
        "gold_rows": gold_count,
    }

accepted, pseudo_audit = deterministic_cap(
    pseudo_predictions,
    CONFIDENCE_THRESHOLD,
)
accepted_95, sensitivity_audit = deterministic_cap(
    pseudo_predictions,
    SENSITIVITY_THRESHOLD,
)
if accepted.count() == 0:
    raise RuntimeError(
        "No balanced pseudo-label pool remained after confidence filtering and caps"
    )


In [4]:
label_counts = {
    int(row["label"]): int(row["count"])
    for row in train.groupBy("label").count().collect()
}
gold_total = sum(label_counts.values())
class_weights = {
    label: gold_total / (2.0 * count)
    for label, count in label_counts.items()
}

def add_training_weight(frame):
    class_weight = (
        F.when(F.col("label") == 0, F.lit(class_weights[0]))
        .otherwise(F.lit(class_weights[1]))
    )
    source_weight = (
        F.when(F.col("label_source") == "pseudo", F.lit(PSEUDO_SOURCE_WEIGHT))
        .otherwise(F.lit(1.0))
    )
    return frame.withColumn("sample_weight", class_weight * source_weight)

pseudo_columns = [
    "review_id", "safe_review", "model_text", "tokens",
    "source_dataset", "features_linear", "features_tree", "features_nb",
    "features_tfidf_only", "features_linear_no_near",
    "handcrafted_raw", "handcrafted_scaled", "handcrafted_nonnegative",
]
pseudo_train = (
    accepted.select(
        *pseudo_columns,
        F.col("pseudo_label").alias("label"),
        F.lit("pseudo_train").alias("source_split"),
        F.lit("pseudo").alias("label_source"),
        "confidence_score",
    )
)
gold_train = train
expanded_train = add_training_weight(
    gold_train.unionByName(pseudo_train, allowMissingColumns=True)
)
gold_weighted = add_training_weight(gold_train)

accepted.select(
    "review_id", "safe_review", "pseudo_label", "confidence_score",
    "source_dataset",
).write.mode("overwrite").parquet(
    str(PROCESSED_DIR / "accepted_pseudo_labels.parquet")
)

expanded_models = {}
for name, gold_model in gold_models.items():
    estimator = estimator_from_gold(name, gold_model)
    expanded_model = estimator.fit(expanded_train)
    expanded_models[name] = expanded_model
    expanded_model.write().overwrite().save(
        str(MODELS_DIR / f"{name}_expanded")
    )


26/09/20 15:14:19 WARN DAGScheduler: Broadcasting large task binary with size 1069.0 KiB
26/09/20 15:14:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/09/20 15:14:21 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/09/20 15:14:26 WARN DAGScheduler: Broadcasting large task binary with size 1658.2 KiB
26/09/20 15:14:29 WARN DAGScheduler: Broadcasting large task binary with size 1658.8 KiB
26/09/20 15:14:30 WARN DAGScheduler: Broadcasting large task binary with size 1658.8 KiB
26/09/20 15:14:30 WARN DAGScheduler: Broadcasting large task binary with size 1658.8 KiB
26/09/20 15:14:30 WARN DAGScheduler: Broadcasting large task binary with size 1658.8 KiB
26/09/20 15:14:31 WARN DAGScheduler: Broadcasting large task binary with size 1658.8 KiB
26/09/20 15:14:31 WARN DAGScheduler: Broadcasting large task binary with size 1658.8 KiB
26/09/20 15:1

In [5]:
comparison_rows = []
for name in gold_models:
    gold_metrics = evaluate_predictions(
        gold_models[name].transform(validation),
        f"{name}_gold",
    )
    expanded_metrics = evaluate_predictions(
        expanded_models[name].transform(validation),
        f"{name}_expanded",
    )
    comparison_rows.append({
        "model": name,
        "gold_macro_f1": gold_metrics["macro_f1"],
        "expanded_macro_f1": expanded_metrics["macro_f1"],
        "delta_macro_f1": expanded_metrics["macro_f1"] - gold_metrics["macro_f1"],
        "gold_fake_pr_auc": gold_metrics["fake_pr_auc"],
        "expanded_fake_pr_auc": expanded_metrics["fake_pr_auc"],
        "delta_fake_pr_auc": (
            expanded_metrics["fake_pr_auc"] - gold_metrics["fake_pr_auc"]
        ),
        "gold_fake_f1": gold_metrics["fake_f1"],
        "expanded_fake_f1": expanded_metrics["fake_f1"],
    })

comparison = pd.DataFrame(comparison_rows).sort_values(
    ["expanded_macro_f1", "expanded_fake_pr_auc"],
    ascending=False,
)
comparison.to_csv(
    RESULTS_DIR / "gold_vs_expanded_validation.csv",
    index=False,
)

selected_model = str(comparison.iloc[0]["model"])
final_manifest = {
    "selected_model": selected_model,
    "selected_model_type": selected_model,
    "selection_source": "gold validation split",
    "gold_model_path": f"models/{selected_model}_gold",
    "expanded_model_path": f"models/{selected_model}_expanded",
    "primary_metric": "expanded_macro_f1",
    "tie_break_metric": "expanded_fake_pr_auc",
}
with open(MODELS_DIR / "final_selection.json", "w", encoding="utf-8") as handle:
    json.dump(final_manifest, handle, indent=2)


26/09/20 15:18:09 WARN DAGScheduler: Broadcasting large task binary with size 1182.5 KiB
26/09/20 15:18:10 WARN DAGScheduler: Broadcasting large task binary with size 1182.5 KiB
26/09/20 15:18:11 WARN DAGScheduler: Broadcasting large task binary with size 1965.3 KiB
26/09/20 15:18:12 WARN DAGScheduler: Broadcasting large task binary with size 1924.6 KiB


In [6]:
best_logistic = gold_models["logistic_regression"]
logistic_parameters = {
    "maxIter": int(model_parameter(best_logistic, "maxIter")),
    "regParam": float(model_parameter(best_logistic, "regParam")),
    "elasticNetParam": float(model_parameter(best_logistic, "elasticNetParam")),
}

def fit_ablation(frame, feature_column):
    return LogisticRegression(
        featuresCol=feature_column,
        labelCol="label",
        weightCol="sample_weight",
        family="binomial",
        **logistic_parameters,
    ).fit(frame)

ablation_rows = []
for condition, frame in (
    ("gold", gold_weighted),
    ("expanded", expanded_train),
):
    for feature_set, feature_column in (
        ("full", "features_linear"),
        ("tfidf_only", "features_tfidf_only"),
        ("without_near_duplicate", "features_linear_no_near"),
    ):
        model = fit_ablation(frame, feature_column)
        metrics = evaluate_predictions(
            model.transform(validation),
            f"{condition}_{feature_set}",
        )
        ablation_rows.append({
            "condition": condition,
            "feature_set": feature_set,
            "macro_f1": metrics["macro_f1"],
            "fake_pr_auc": metrics["fake_pr_auc"],
            "fake_f1": metrics["fake_f1"],
        })

ablation = pd.DataFrame(ablation_rows)
ablation.to_csv(RESULTS_DIR / "feature_ablation_validation.csv", index=False)

sensitivity_result = None
if accepted_95.count() > 0:
    pseudo_95 = accepted_95.select(
        *pseudo_columns,
        F.col("pseudo_label").alias("label"),
        F.lit("pseudo_train_95").alias("source_split"),
        F.lit("pseudo").alias("label_source"),
        "confidence_score",
    )
    expanded_95 = add_training_weight(
        gold_train.unionByName(pseudo_95, allowMissingColumns=True)
    )
    logistic_95 = fit_ablation(expanded_95, "features_linear")
    sensitivity_result = evaluate_predictions(
        logistic_95.transform(validation),
        "logistic_regression_threshold_095",
    )


26/09/20 15:18:51 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/09/20 15:18:54 WARN DAGScheduler: Broadcasting large task binary with size 1658.4 KiB
26/09/20 15:18:54 WARN DAGScheduler: Broadcasting large task binary with size 1658.9 KiB
26/09/20 15:18:55 WARN DAGScheduler: Broadcasting large task binary with size 1658.9 KiB
26/09/20 15:18:55 WARN DAGScheduler: Broadcasting large task binary with size 1658.9 KiB
26/09/20 15:18:56 WARN DAGScheduler: Broadcasting large task binary with size 1658.9 KiB
26/09/20 15:18:56 WARN DAGScheduler: Broadcasting large task binary with size 1658.9 KiB
26/09/20 15:18:56 WARN DAGScheduler: Broadcasting large task binary with size 1658.9 KiB
26/09/20 15:18:57 WARN DAGScheduler: Broadcasting large task binary with size 1658.9 KiB
26/09/20 15:18:57 WARN DAGScheduler: Broadcasting large task binary with size 1658.9 KiB
26/09/20 15:18:57 WARN DAGScheduler: Broadcasting large task binary with size 1658.9 KiB
26/09/20 15:18:58 WARN D

In [7]:
pseudo_label_audit = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "confidence_rule": "strictly greater than threshold",
    "primary": pseudo_audit,
    "sensitivity": sensitivity_audit,
    "pseudo_source_weight": PSEUDO_SOURCE_WEIGHT,
    "class_weights_from_gold": class_weights,
    "validation_comparison": comparison_rows,
    "feature_ablation": ablation_rows,
    "sensitivity_result": sensitivity_result,
    "final_selection": final_manifest,
}
with open(AUDIT_DIR / "pseudo_label_audit.json", "w", encoding="utf-8") as handle:
    json.dump(pseudo_label_audit, handle, ensure_ascii=False, indent=2)

print(comparison.to_string(index=False))
print(ablation.to_string(index=False))
print(json.dumps(final_manifest, indent=2))


              model  gold_macro_f1  expanded_macro_f1  delta_macro_f1  gold_fake_pr_auc  expanded_fake_pr_auc  delta_fake_pr_auc  gold_fake_f1  expanded_fake_f1
      random_forest       0.923148           0.924887        0.001740          0.913686              0.921245           0.007559      0.867209          0.870270
                gbt       0.923825           0.924855        0.001029          0.913083              0.901996          -0.011088      0.869110          0.870712
logistic_regression       0.911465           0.914099        0.002633          0.902751              0.906673           0.003922      0.848329          0.852713
         linear_svc       0.776893           0.804842        0.027949          0.659772              0.689428           0.029656      0.612360          0.661064
        naive_bayes       0.704617           0.707029        0.002411          0.536721              0.536729           0.000008      0.520548          0.524462
condition            feature_set  